# 14. Från Python till R

Statistiska analyser görs ofta i **R**, och resultat i rapporter och artiklar presenteras gärna som R-utskrifter.
Du behöver inte kunna programmera i R för den här kursen, men du ska kunna **läsa en R-utskrift** och känna igen
det du redan har lärt dig.

Den goda nyheten: statistiken är densamma. Bara stavningen skiljer.

**Efter den här notebooken kan du**
- para ihop kursens analyser i Python med motsvarande kommandon i R
- läsa utskriften från `t.test` och `summary(lm(...))` i R
- hitta testvariabel, frihetsgrader, p-värde och konfidensintervall i en utskrift
- förklara varför R och Python ibland ger lite olika p-värden

**Förkunskaper:** [8. Hypotesprövning](../08_hypotesprovning/08_hypotesprovning.ipynb) och
[13. Enkel linjär regression](../13_linjar_regression/13_linjar_regression.ipynb)

> Notebooken körs med Python. R-koden visas som text. Har du R installerat kan du klistra in den där och jämföra.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

## 14.1 Ordlista: Python ↔ R

| Analys | Python (scipy) | R | Notebook |
|---|---|---|---|
| Medelvärde, standardavvikelse | `x.mean()`, `x.std(ddof=1)` | `mean(x)`, `sd(x)` | 3 |
| Kvartiler, översikt | `x.describe()` | `summary(x)` | 3 |
| Andel under ett värde i normalfördelningen | `stats.norm.cdf(x, mu, sigma)` | `pnorm(x, mu, sigma)` | 4 |
| Kritiskt värde, t-fördelningen | `stats.t.ppf(0.975, df=n-1)` | `qt(0.975, df = n-1)` | 7 |
| t-test, ett stickprov | `stats.ttest_1samp(x, 170)` | `t.test(x, mu = 170)` | 8 |
| t-test, två grupper (Welch) | `stats.ttest_ind(a, b, equal_var=False)` | `t.test(a, b)` | 8 |
| Två andelar | manuellt z-test | `prop.test(c(115, 150), c(1000, 1000))` | 9 |
| Korstabell | `pd.crosstab(a, b)` | `table(a, b)` | 11 |
| Chi-två-test | `stats.chi2_contingency(table)` | `chisq.test(table)` | 11 |
| Pearsons korrelation | `stats.pearsonr(x, y)` | `cor.test(x, y)` | 12 |
| Linjär regression | `stats.linregress(x, y)` | `summary(lm(y ~ x))` | 13 |

Några skillnader i stavning som är bra att känna till:

| | Python | R |
|---|---|---|
| Tilldelning | `x = 5` | `x <- 5` |
| Skapa en talserie | `np.array([1, 2, 3])` | `c(1, 2, 3)` |
| Argument med namn | `equal_var=False` | `var.equal = FALSE` |
| Sant/falskt | `True`, `False` | `TRUE`, `FALSE` |
| Formel "y förklaras av x" | – | `y ~ x` |

## 14.2 Läsa utskriften från ett t-test

I [notebook 8](../08_hypotesprovning/08_hypotesprovning.ipynb) jämförde du handläggningstider med en gammal och en ny rutin.
I R skulle samma analys se ut så här:

```r
old_routine <- c(34, 41, 38, 45, 36, 40, 43, 39, 37, 42, 44, 35)
new_routine <- c(31, 36, 33, 39, 30, 35, 38, 34, 37, 32, 36, 33)
t.test(old_routine, new_routine)
```

och ge en utskrift av det här slaget:

```
	Welch Two Sample t-test

data:  old_routine and new_routine
t = 3.7879, df = 20.769, p-value = 0.001095
alternative hypothesis: true difference in means is not equal to 0
95 percent confidence interval:
 2.253029 7.746971
sample estimates:
mean of x mean of y
     39.5      34.5
```

Rad för rad:

| Rad i utskriften | Betyder |
|---|---|
| `Welch Two Sample t-test` | Vilket test som gjordes. R använder Welch som standard, precis som vi gjort med `equal_var=False`. |
| `t = 3.7879` | Testvariabeln: skillnaden delad med sitt medelfel. |
| `df = 20.769` | Frihetsgrader. I Welchs test räknas de ut med en formel och blir sällan ett heltal. |
| `p-value = 0.001095` | p-värdet. |
| `alternative hypothesis: ... not equal to 0` | $H_1$. "Not equal" betyder dubbelsidigt test. |
| `95 percent confidence interval` | KI för skillnaden mellan medelvärdena (x − y). |
| `sample estimates` | De två stickprovsmedelvärdena. |

✏️ **Din tur:** Ta fram *varje tal* i R-utskriften med Python: t, df, p-värde, konfidensintervallets gränser och de två medelvärdena.
Ledtråd: resultatet från `stats.ttest_ind` har attributen `.statistic`, `.df` och `.pvalue` samt metoden `.confidence_interval(0.95)`.

In [ ]:
old_routine = np.array([34, 41, 38, 45, 36, 40, 43, 39, 37, 42, 44, 35])
new_routine = np.array([31, 36, 33, 39, 30, 35, 38, 34, 37, 32, 36, 33])

# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
res = stats.ttest_ind(old_routine, new_routine, equal_var=False)
ci = res.confidence_interval(0.95)
print(f"t = {res.statistic:.4f}, df = {res.df:.3f}, p-value = {res.pvalue:.6f}")
print(f"95 % KI: {ci.low:.6f} {ci.high:.6f}")
print(f"mean of x = {old_routine.mean()}, mean of y = {new_routine.mean()}")
```

</details>

✏️ **Din tur:** I notebook 8 gjorde du testet *ensidigt* och fick ett annat p-värde. I R skrivs det
`t.test(old_routine, new_routine, alternative = "greater")`. Vad blir p-värdet då? Räkna ut det i Python,
och förklara hur det hänger ihop med 0,001095.

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
res_greater = stats.ttest_ind(old_routine, new_routine, equal_var=False, alternative="greater")
print(f"Ensidigt p-värde: {res_greater.pvalue:.6f}")
# Hälften av det dubbelsidiga, eftersom skillnaden går åt det håll som H1 pekar ut.
```

</details>

## 14.3 Läsa utskriften från en regression

Regressionsutskriften är den R-utskrift du oftast kommer att stöta på. Så här ser den ut för en regression av provpoäng
mot studietimmar, på data av samma slag som i [notebook 13](../13_linjar_regression/13_linjar_regression.ipynb)
(50 studenter, men ett annat slumpmässigt stickprov, så siffrorna skiljer sig lite från dina).

```r
summary(lm(exam_score ~ study_hours))
```

```
Coefficients:
            Estimate Std. Error t value Pr(>|t|)
(Intercept)  38.2405     2.0288   18.85   <2e-16 ***
study_hours   2.4846     0.1754   14.16   <2e-16 ***
---
Signif. codes:  0 '***' 0.001 '**' 0.01 '*' 0.05 '.' 0.1 ' ' 1

Multiple R-squared:  0.8069,	Adjusted R-squared:  0.8029
```

Varje **rad** i tabellen är en koefficient: `(Intercept)` är *a*, och raden med variabelns namn är lutningen *b*.

| Kolumn | Betyder | I `linregress` |
|---|---|---|
| `Estimate` | Koefficientens skattade värde | `.intercept`, `.slope` |
| `Std. Error` | Koefficientens medelfel | `.intercept_stderr`, `.stderr` |
| `t value` | Estimate / Std. Error | – |
| `Pr(>\|t\|)` | p-värdet för $H_0$: koefficienten är 0 | `.pvalue` (för lutningen) |

- **Stjärnorna** är en snabbmarkering av p-värdet enligt raden *Signif. codes*: tre stjärnor betyder p < 0,001.
- `<2e-16` betyder "mindre än 0,000 000 000 000 000 2", alltså så litet att R inte skriver ut det.
- **Multiple R-squared** är R². *Adjusted R-squared* är en variant som justerar för antalet förklarande variabler och
  som blir viktig först när modellen har flera x-variabler.

🔮 **Gissa först:** skriv regressionsekvationen utifrån utskriften. Vad är prognosen för en student som pluggar 10 timmar?

✏️ **Din tur:** Använd bara talen i utskriften (och Python som miniräknare) för att
1. räkna ut prognosen för 10 studietimmar,
2. kontrollera `t value` för `study_hours`,
3. räkna ut p-värdet för `study_hours` (dubbelsidigt, med n − 2 = 48 frihetsgrader) – stämmer det att det är mindre än 2e-16?
4. göra ett 95 % konfidensintervall för lutningen,
5. räkna ut korrelationskoefficienten r.

In [ ]:
estimate_a, se_a = 38.2405, 2.0288
estimate_b, se_b = 2.4846, 0.1754
r_squared = 0.8069

# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
print(f"1. Prognos för 10 timmar: {estimate_a + estimate_b * 10:.1f} poäng")
t_value = estimate_b / se_b
print(f"2. t value = {t_value:.2f}")
print(f"3. p-värde = {2 * stats.t.sf(t_value, df=48):.1e}")
t_crit = stats.t.ppf(0.975, df=48)
print(f"4. 95 % KI för lutningen: [{estimate_b - t_crit * se_b:.2f}, {estimate_b + t_crit * se_b:.2f}]")
print(f"5. r = {np.sqrt(r_squared):.2f}  (positivt, eftersom lutningen är positiv)")
```

</details>

## 14.4 När R och Python inte ger samma svar

Ibland ger samma test lite olika p-värden i olika program. Det beror nästan alltid på **olika standardinställningar**,
inte på att något är fel.

Det vanligaste exemplet är **kontinuitetskorrektionen**. A/B-testet i
[notebook 9](../09_tva_andelar/09_tva_andelar.ipynb) gav p = 0,0210 med vårt z-test. I R ger

```r
prop.test(c(115, 150), c(1000, 1000))
```

i stället `X-squared = 5.0285, df = 1, p-value = 0.02493`. Skillnaden är att `prop.test` som standard lägger på en
korrektion som gör testet något försiktigare. scipys `chi2_contingency` gör likadant för tabeller med 2 × 2 celler.

In [ ]:
table = [[115, 885],
         [150, 850]]

for correction in [True, False]:
    res = stats.chi2_contingency(table, correction=correction)
    print(f"correction={correction!s:<5}  χ² = {res.statistic:.4f}   p-värde = {res.pvalue:.5f}")

Med korrektionen får vi R:s svar, utan den får vi z-testets. (I R stänger man av den med `correct = FALSE`.)

Andra standardval som skiljer sig:

| | Python | R |
|---|---|---|
| t-test för två grupper | `stats.ttest_ind` antar **lika varianser** om man inte skriver `equal_var=False` | `t.test` använder **Welch** om man inte skriver `var.equal = TRUE` |
| Standardavvikelse | `np.std(x)` delar med **n**, pandas `.std()` med **n − 1** | `sd(x)` delar med **n − 1** |

✏️ **Din tur:** Någon kör `stats.ttest_ind(old_routine, new_routine)` i Python *utan* `equal_var=False` och får inte samma
svar som R. Kör det själv. Vilka tal i utskriften ändras, och hur skulle man få R att ge samma svar?

In [ ]:
# Skriv din kod här

<details>
<summary>💡 <b>Lösningsförslag</b> (försök själv först!)</summary>

```python
res_pooled = stats.ttest_ind(old_routine, new_routine)
print(f"t = {res_pooled.statistic:.4f}, df = {res_pooled.df:.0f}, p-value = {res_pooled.pvalue:.6f}")
# t-värdet är detsamma här (grupperna är lika stora), men frihetsgraderna blir 22 i stället för 20,769
# och p-värdet ändras därför något. I R: t.test(old_routine, new_routine, var.equal = TRUE).
```

</details>

> **Lärdom:** ange alltid *vilket* test du har använt, med vilka inställningar. "Vi gjorde ett t-test" räcker inte.

## Sammanfattning

- R och Python gör samma statistik med olika stavning.
- I en R-utskrift letar du efter: **testets namn**, **testvariabeln** (t, X-squared), **df**, **p-value**, **konfidensintervall** och **skattningar**.
- I en regressionsutskrift: **Estimate** (koefficienter), **Std. Error** (medelfel), **t value**, **Pr(>|t|)** (p-värde) och **R-squared**.
- Olika program har olika standardinställningar. Redovisa vilket test och vilka val du gjort.

### Kontrollfrågor
1. En R-utskrift visar `study_hours  0.4210  0.3105  1.356  0.181`. Finns det belägg för ett samband?
2. Vad betyder `**` efter ett p-värde i en R-utskrift?
3. En R-utskrift från `t.test` visar `df = 37.4`. Hur kan antalet frihetsgrader vara ett decimaltal?

<details>
<summary>💡 <b>Svar</b></summary>

1. Nej. p-värdet 0,181 är större än 0,05, så lutningen skiljer sig inte signifikant från 0.
2. Att p-värdet ligger mellan 0,001 och 0,01.
3. Det är Welchs t-test, där frihetsgraderna räknas ut med en formel som tar hänsyn till att grupperna kan ha olika
   varians och storlek.

</details>

**Nästa notebook:** [15. Kritisk värdering av slutsatser](../15_kritisk_vardering/15_kritisk_vardering.ipynb)